# Phase 1 — LA-basin PM2.5 → building DALYs (local prototype)

End-to-end demonstration of the analytical chain that the cloud phases
will scale up:

1. Load staged AirNow hourly observations (LA basin AOI) + Overture
   building centroids.
2. Hourly → daily mean per monitor; compute completeness.
3. Annual mean / 98th-percentile day / peak-week per monitor.
4. IDW interpolation of annual mean PM2.5 to every building centroid.
5. Concentration-response → expected annual DALYs per building, per
   cause (IHD, stroke, COPD, lung cancer, LRI).
6. Aggregate to H3 r8 cells; render folium choropleth.

All heavy lifting lives in `airhealth.{features,io,scoring}` so the
Phase 2 Sedona job can reuse the same primitives unchanged.

**Note on coverage.** This notebook runs on whatever AirNow days you've
staged under `data/raw/airnow/`. With only a partial window staged,
"annual" metrics are really window-mean — fine for plumbing validation,
not for headline numbers.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config" / "release.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from airhealth.ingest._common import load_release_config
from airhealth.features import annual_metrics, assign_h3_cells, completeness, daily_mean
from airhealth.io import (
    aggregate_to_h3,
    read_airnow_window,
    read_overture_centroids,
    write_folium_choropleth,
)
from airhealth.scoring import (
    expected_annual_dalys,
    idw_interpolate,
    load_concentration_response,
)

cfg = load_release_config()
cr = load_concentration_response(REPO_ROOT / "config" / "concentration_response.yaml")
print("AOI:", cfg.aoi.name, cfg.aoi.bbox)
print("AirNow window:", cfg.airnow_window)
print("CR causes:", [c.key for c in cr.causes])

AOI: la_basin (-118.95, 33.7, -117.65, 34.35)
AirNow window: 20250101-20251231
CR causes: ['ihd', 'stroke', 'copd', 'lung_cancer', 'lri']


## 1. Load AirNow hourly observations

In [2]:
hourly = read_airnow_window(cfg.aoi)
print(f"hourly rows: {len(hourly):,}  monitors: {hourly['aqsid'].nunique()}")
hourly.head(3)

hourly rows: 1,509  monitors: 9


,valid_date,valid_time,aqsid,site_name_x,gmt_offset,parameter,units,value,data_source,lat,lon,site_name_y,state_code
0,01/01/25,00:00,060370016,Glendora - Laurel,-8.0,PM2.5,UG/M3,26.9,South Coast AQMD,34.143900,-117.850800,Glendora - Laurel,CA
1,01/01/25,00:00,060371103,Los Angeles - N. Mai,-8.0,PM2.5,UG/M3,30.4,South Coast AQMD,34.066429,-118.226755,Los Angeles - N. Main Street,CA
2,01/01/25,00:00,060371201,Reseda,-8.0,PM2.5,UG/M3,28.0,South Coast AQMD,34.199200,-118.533100,Reseda,CA


## 2. Daily means + completeness per monitor

In [3]:
daily = daily_mean(hourly)
comp = completeness(daily)
print(f"daily rows: {len(daily):,}  unique dates: {daily['date_utc'].nunique()}")
comp.sort_values("completeness", ascending=False)

daily rows: 63  unique dates: 7


,aqsid,lat,lon,n_hours,completeness
0,060370016,34.143900,-117.850800,168,0.019178
1,060371103,34.066429,-118.226755,168,0.019178
2,060371201,34.199200,-118.533100,168,0.019178
3,060371302,33.901400,-118.205000,168,0.019178
4,060590007,33.830586,-117.938509,168,0.019178
6,061112002,34.276320,-118.683690,168,0.019178
8,840060374010,34.181977,-118.363036,168,0.019178
7,840060374009,33.793713,-118.171019,167,0.019064
5,061110007,34.210170,-118.870510,166,0.018950


## 3. Annual exposure metrics per monitor

`annual_mean` feeds the chronic-exposure DALY calc; `p98_day` and
`peak_week_mean` are kept for the dashboard's acute-tail panel.


In [4]:
monitors = annual_metrics(daily)
monitors

,aqsid,lat,lon,annual_mean,p98_day,peak_week_mean,n_days,n_hours,completeness
0,060370016,34.143900,-117.850800,11.525595,25.840000,15.748958,7,168,0.019178
1,060371103,34.066429,-118.226755,21.198810,36.568167,28.375000,7,168,0.019178
2,060371201,34.199200,-118.533100,15.845833,52.670833,24.548958,7,168,0.019178
3,060371302,33.901400,-118.205000,28.327381,54.546333,35.109375,7,168,0.019178
4,060590007,33.830586,-117.938509,27.230952,55.722000,35.671875,7,168,0.019178
5,061110007,34.210170,-118.870510,6.096532,15.480000,9.052083,7,166,0.018950
6,061112002,34.276320,-118.683690,5.630952,10.720000,8.052083,7,168,0.019178
7,840060374009,33.793713,-118.171019,25.339027,50.146239,30.846422,7,167,0.019064
8,840060374010,34.181977,-118.363036,22.902976,52.627167,30.860417,7,168,0.019178


## 4. Load Overture building centroids

3.5 M LA-basin buildings is too many for folium; we sample for the
choropleth but keep the IDW + DALY math vectorized at the sample size.
The Spark job (Phase 2) runs the same math at full scale.


In [5]:
OVERTURE_PARQUET = REPO_ROOT / "data" / "raw" / "overture_la_basin_2026-04-15.0.parquet.uploaded"
SAMPLE_BUILDINGS = 50_000  # ~50 K is plenty for an H3-r8 choropleth.

buildings = read_overture_centroids(OVERTURE_PARQUET, sample=SAMPLE_BUILDINGS)
print(f"buildings sampled: {len(buildings):,}")
buildings.head(3)

buildings sampled: 50,000


,id,class,num_floors,height,area_m2,lon,lat
0,321188f3-c676-4bcd-95ec-7e88640de8b6,house,<NA>,4.400000,10.377059,-118.561476,34.185433
1,2b5a6686-0ca7-46a2-badb-f3dba854794f,house,<NA>,4.300000,224.590047,-117.898209,34.105133
2,49d45ff2-05ee-403c-8e1f-06f948208c13,NaN,<NA>,4.232509,357.787319,-118.728664,34.287746


## 5. IDW: monitor PM2.5 → building PM2.5

Power 2 (Shepard default), max neighbor radius 150 km (CONUS rural
gaps); LA basin has 9 monitors so every building has plenty.


In [6]:
buildings["pm25_ugm3"] = idw_interpolate(
    buildings["lon"].to_numpy(),
    buildings["lat"].to_numpy(),
    monitors["lon"].to_numpy(),
    monitors["lat"].to_numpy(),
    monitors["annual_mean"].to_numpy(),
    power=2.0,
    max_km=150.0,
)
buildings["pm25_ugm3"].describe()

count    50000.000000
mean        21.360592
std          4.565136
min          5.631184
25%         19.447075
50%         21.834159
75%         25.080632
max         28.326247
Name: pm25_ugm3, dtype: float64

## 6. Per-building DALYs

Population estimate uses Overture `num_floors × area_m²` × a crude
occupants-per-m² constant (refined in Phase 4 with ACS). Folded into
the CR × baseline-mortality math from `airhealth.scoring`.


In [7]:
# Crude per-building occupancy: 0.04 occupants/m² floor area
# (≈ US residential average of 25 m²/person, mixes commercial in).
buildings["floor_area_m2"] = (
    buildings["area_m2"].fillna(0)
    * buildings["num_floors"].fillna(1).clip(lower=1)
)
buildings["population"] = buildings["floor_area_m2"] * 0.04

daly_by_cause = expected_annual_dalys(
    buildings["pm25_ugm3"].to_numpy(),
    buildings["population"].to_numpy(),
    cr,
)
for k, v in daly_by_cause.items():
    buildings[f"daly_{k}"] = v
buildings["daly_total"] = sum(daly_by_cause.values())

print("Total DALYs in sample:", buildings["daly_total"].sum())
print("Per-cause share:")
for k in daly_by_cause:
    share = buildings[f"daly_{k}"].sum() / buildings["daly_total"].sum()
    print(f"  {k:12s} {share:.1%}")

Total DALYs in sample: 4374.323316749573
Per-cause share:
  ihd          43.5%
  stroke       12.7%
  copd         13.4%
  lung_cancer  20.4%
  lri          10.0%


## 7. H3 indexing + aggregation for the choropleth

In [8]:
buildings = assign_h3_cells(buildings)
cells_r8 = aggregate_to_h3(
    buildings,
    h3_col="h3_r8",
    value_cols=("daly_total", "population"),
)
cells_r8["dalys_per_capita"] = cells_r8["daly_total"] / cells_r8["population"].replace(0, np.nan)
print(f"H3 r8 cells: {len(cells_r8):,}")
cells_r8.sort_values("daly_total", ascending=False).head()

H3 r8 cells: 5,185


,h3_r8,daly_total,population,n_buildings,dalys_per_capita
2490,8829a19989fffff,77.996217,11661.153481,4,0.006689
3591,8829a1d5c3fffff,34.590430,4920.877751,10,0.007029
3630,8829a1d62dfffff,24.782908,3507.270202,5,0.007066
3490,8829a1d49dfffff,22.330986,3037.959329,7,0.007351
3742,8829a1d753fffff,17.572849,2493.006566,3,0.007049


## 8. Folium choropleth — annual DALYs per H3 r8 cell

In [9]:
OUT = REPO_ROOT / "data" / "la_basin_dalys_r8.html"
write_folium_choropleth(
    cells_r8,
    value_col="daly_total",
    h3_col="h3_r8",
    out_html=OUT,
    aoi=cfg.aoi,
    legend="Expected annual DALYs (per H3 r8 cell)",
)
print(f"wrote {OUT}  ({OUT.stat().st_size/1024:.0f} KB)")
print("open in a browser:  open", OUT)

wrote /Users/syzheng/Documents/learn/UrbanAirHealth/data/la_basin_dalys_r8.html  (2478 KB)
open in a browser:  open /Users/syzheng/Documents/learn/UrbanAirHealth/data/la_basin_dalys_r8.html


## Caveats (Phase 1)

- **Coverage:** if only a partial window is staged under
  `data/raw/airnow/`, every "annual" number is a window-mean. Pull the
  full year via `python -m airhealth.ingest.airnow ...` before quoting.
- **Population:** flat 0.04 occupants/m² is a crude stand-in; Phase 4
  replaces with ACS tract population × HUD residential mask.
- **Sample bias:** we choropleth a 50 K-building reservoir sample. The
  Spark job runs the same math at full 3.5 M scale.
- **Log-linear CR:** reliable up to ~50 µg/m³; sublinear above (wildfire
  smoke). Flag `pm25_ugm3 > 50` rows; Phase 4 swaps for full IER.
